# Thermal de Broglie wavelength

## Prerequisites

Read the [analytical derivation](../../../../docs/lecture-notes/statistical-mechanics/thermal-de-broglie-wavelength/index.md). Review the de Broglie relation, the canonical partition function, Gaussian integration, and numerical quadrature.

## Objectives

1. Recover the thermal de Broglie wavelength from a numerical momentum-space Gaussian integral.
2. Compare that computational derivation with the package-owned analytical evaluator.
3. Verify the $T^{-1/2}$ scaling over a temperature interval.
4. Evaluate the unitless parameter $n\lambda_T^3$ without imposing a regime threshold.

The thermal wavelength is a statistical scale for an ensemble. It is not one particle's definite wavelength or the literal spatial extent of a wave packet.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

from projectkoios.physkit.constants import SI
from projectkoios.physkit.thermal.statmech.thermal_wavelength import (
    ThermalDeBroglieWavelengthModel,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

## Computational derivation

For one Cartesian momentum component,

$$
I_p=\int_{-\infty}^{\infty}\exp\!\left(-\frac{p^2}{2mk_BT}\right)dp.
$$

With $u=p/\sqrt{2mk_BT}$, numerical quadrature supplies the dimensionless Gaussian factor while the physical momentum scale remains explicit. Since $Z_1/V=(I_p/h)^3$, the computed normalization length is $\lambda_T=h/I_p$.


In [ ]:
electron_mass = ScalarQuantity(
    magnitude=float(SI.me0),
    unit=PhysicalUnit(expression="kilogram"),
)
reference_temperature = ScalarQuantity(
    magnitude=300.0,
    unit=PhysicalUnit(expression="kelvin"),
)
model = ThermalDeBroglieWavelengthModel(particle_mass=electron_mass)

negative_infinity = -np.inf
positive_infinity = np.inf
gaussian_integral, gaussian_integral_error = quad(
    lambda dimensionless_momentum: np.exp(-(dimensionless_momentum**2)),
    negative_infinity,
    positive_infinity,
)
thermal_momentum_scale = math.sqrt(
    2.0 * electron_mass.magnitude * SI.k_B * reference_temperature.magnitude
)
momentum_integral = thermal_momentum_scale * gaussian_integral
quadrature_wavelength_metres = SI.h / momentum_integral

analytical_evaluation = model.evaluate_wavelengths(
    temperatures=VectorQuantity(
        magnitude=np.array([reference_temperature.magnitude], dtype=np.float64),
        unit=reference_temperature.unit,
    ),
    wavelength_unit=PhysicalUnit(expression="meter"),
)
analytical_wavelength_metres = analytical_evaluation.wavelengths.magnitude[0]
quadrature_relative_tolerance = 2.0e-14
assert np.isclose(
    quadrature_wavelength_metres,
    analytical_wavelength_metres,
    rtol=quadrature_relative_tolerance,
    atol=0.0,
)
gaussian_absolute_error_limit = 2.0e-8
assert gaussian_integral_error < gaussian_absolute_error_limit

In [ ]:
minimum_temperature_kelvin = 100.0
maximum_temperature_kelvin = 10_000.0
temperature_sample_count = 300
temperatures_kelvin = np.geomspace(
    minimum_temperature_kelvin,
    maximum_temperature_kelvin,
    temperature_sample_count,
).astype(np.float64)
wavelength_evaluation = model.evaluate_wavelengths(
    temperatures=VectorQuantity(
        magnitude=temperatures_kelvin,
        unit=PhysicalUnit(expression="kelvin"),
    ),
    wavelength_unit=PhysicalUnit(expression="nanometer"),
)
logarithmic_slope = np.polyfit(
    np.log(temperatures_kelvin),
    np.log(wavelength_evaluation.wavelengths.magnitude),
    deg=1,
)[0]
scaling_absolute_tolerance = 2.0e-15
assert np.isclose(
    logarithmic_slope,
    -0.5,
    rtol=0.0,
    atol=scaling_absolute_tolerance,
)

In [ ]:
illustrative_number_density = ScalarQuantity(
    magnitude=2.5e25,
    unit=PhysicalUnit(expression="meter ** -3"),
)
degeneracy_evaluation = model.evaluate_degeneracy_parameter(
    number_density=illustrative_number_density,
    temperature=reference_temperature,
    wavelength_unit=PhysicalUnit(expression="nanometer"),
)
assert np.isclose(
    degeneracy_evaluation.wavelength.magnitude,
    4.303475436666189,
    rtol=2.0e-15,
    atol=0.0,
)
assert degeneracy_evaluation.degeneracy_parameter.magnitude >= 0.0

In [ ]:
figure_size_inches = (8.0, 4.8)
figure, axis = plt.subplots(figsize=figure_size_inches)
axis.loglog(
    temperatures_kelvin,
    wavelength_evaluation.wavelengths.magnitude,
    label="electron thermal wavelength",
)
axis.scatter(
    [reference_temperature.magnitude],
    [degeneracy_evaluation.wavelength.magnitude],
    color="tab:red",
    label="300 K checkpoint",
    zorder=3,
)
axis.set(
    xlabel="temperature (K)",
    ylabel="thermal de Broglie wavelength (nm)",
    title="Canonical electron thermal wavelength",
)
axis.grid(True, which="both")
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

The numerical momentum integral and the analytical package evaluator agree at the stated tolerance. The logarithmic slope is $-1/2$, consistent with $\lambda_T\propto T^{-1/2}$. The 300 K electron checkpoint is approximately $4.30\ \mathrm{nm}$ under the canonical partition-function convention.

The calculated value of $n\lambda_T^3$ belongs to an illustrative density and is not a material-specific validation result. No universal classical-versus-quantum threshold is applied.

## Exercises

1. Repeat the calculation for a proton or neutral atom using an explicit mass quantity.
2. Derive the three-dimensional momentum integral as a product of three one-dimensional Gaussian integrals.
3. Change the requested wavelength unit and verify that $n\lambda_T^3$ remains unchanged.
4. Investigate how an internal degeneracy factor modifies state counting without changing the definition of $\lambda_T$.
